# Resume direct attack on contextual recovery (v1)

Bounded direct attack on the **frozen v4** resume recovery rule. The defense is unchanged; only the attack is new.

- **Documents:** the first ten reserved resumes that have saved attacks, in the frozen order (not selected on outcomes).
- **Budgets:** epsilon 8 and 16, two restarts each: 40 runs. All restart-0 runs go first, then restart-1.
- **Per run:** up to 100 gradient steps on the localized and expanded views, re-localizing every 10 steps, plus random-patch verifier queries, 40 verifier queries in total. A run stops early once the recovery rule accepts the target.
- **Success:** the saved, reloaded PNG pages make both views agree on the target label.
- **Session limit:** stops starting new runs after about 3.5 hours. Run all cells again to continue; finished runs are kept.

Place this notebook in `MyDrive/CSCI 566 Project/test/csev_full_colab/resume_colab/`. It needs these files, which are already on Drive:
`resume_colab/resume_development_inputs_v1.zip`, `resume_colab/resume_complete_inputs_v1.zip`, `resume_colab/resume_reserved_results_v1/` and `csev_full_colab/source.tar`.

Select an A100 runtime and use **Runtime > Run all**. Results go to `resume_colab/resume_direct_attack_v1/`. Missing runs are never counted as attack failures. This is not a robustness guarantee.

## 1. Mount Drive and prepare verified inputs

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, hashlib, shutil, zipfile, tarfile, tempfile, os
from datetime import datetime

drive.mount('/content/drive')
RESUME_DRIVE = Path('/content/drive/MyDrive/CSCI 566 Project/test/csev_full_colab/resume_colab')
DEV_BUNDLE = RESUME_DRIVE / 'resume_development_inputs_v1.zip'
COMPLETE = RESUME_DRIVE / 'resume_complete_inputs_v1.zip'
SOURCE_TAR = RESUME_DRIVE.parent / 'source.tar'
RESERVED = RESUME_DRIVE / 'resume_reserved_results_v1'
ATTACK_ROOT = Path('/content/resume_direct_attack_v1')
ATTACK_OUT = RESUME_DRIVE / 'resume_direct_attack_v1'
for needed in (DEV_BUNDLE, COMPLETE, SOURCE_TAR, RESERVED / 'summary.json', RESERVED / 'cases'):
    assert needed.exists(), f'Missing on Drive: {needed}'
subprocess.run(['nvidia-smi'], check=True)

def sha_bytes(b): return hashlib.sha256(b).hexdigest()
def sha_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

EXPECTED_IDS = ['resume_parsing_vision/validation/' + x for x in
    ('r00474','r00373','r00291','r00496','r00842','r00793','r00201','r00063','r00599','r00288')]

if not (ATTACK_ROOT / 'READY.json').exists():
    assert not ATTACK_ROOT.exists(), 'Partial setup found. Use Runtime > Disconnect and delete runtime, then run all again.'
    temp = Path(tempfile.mkdtemp(prefix='resume_attack_', dir='/content'))
    hashes = {}
    def put(name, data):
        target = temp / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
        hashes[name] = sha_bytes(data)

    # 1. Frozen configuration (same file the reserved evaluation used).
    with zipfile.ZipFile(DEV_BUNDLE) as z:
        dev_hashes = json.loads(z.read('hashes.json'))
        config_bytes = z.read('config.json')
        assert sha_bytes(config_bytes) == dev_hashes['config.json']
    put('config.json', config_bytes)
    cfg = json.loads(config_bytes)

    # 2. Tested attack source code from the frozen repository snapshot.
    print('Copying source snapshot from Drive...', flush=True)
    local_tar = temp / 'source.tar'
    shutil.copy2(SOURCE_TAR, local_tar)
    assert sha_file(local_tar) == '1859fd630629d16e9996dd9a4cf61df578a9ef87b27ae059c45f18329a50b261', 'source.tar differs from the tested snapshot'
    with tarfile.open(local_tar) as t:
        for member in t.getmembers():
            name = member.name[2:] if member.name.startswith('./') else member.name
            if member.isfile() and name.startswith('src/') and name.endswith('.py'):
                put(name, t.extractfile(member).read())
    local_tar.unlink()
    for required in ('src/attacks/qwen.py', 'src/attacks/donut.py', 'src/models/qwen.py', 'src/models/donut.py', 'src/checkpoints.py'):
        assert required in hashes, required

    # 3. Completed reserved evaluation: targets, clean status and saved-attack outcomes.
    reserved_summary = json.loads((RESERVED / 'summary.json').read_text())
    assert reserved_summary['complete'] and reserved_summary['cases'] == 296
    print('Reading 296 reserved case records from Drive...', flush=True)
    cases = [json.loads(p.read_text()) for p in sorted((RESERVED / 'cases').glob('*.json'))]
    assert len(cases) == 296
    def status(m): return 'correct' if m['accepted'] and m['correct'] else ('wrong' if m['accepted'] else 'rejected')
    clean = {c['record_id']: c for c in cases if c['family'] == 'clean'}
    saved = {(c['record_id'], c['epsilon_255']): c for c in cases if c['family'] == 'standard'}
    selected = [r for r in cfg['reserved_ids'] if (r, 8) in saved and (r, 16) in saved][:10]
    assert selected == EXPECTED_IDS, selected

    # 4. Clean pages of the ten selected resumes.
    print('Copying complete resume bundle from Drive (1.15 GB)...', flush=True)
    local_zip = temp / 'complete.zip'
    shutil.copy2(COMPLETE, local_zip)
    documents = []
    with zipfile.ZipFile(local_zip) as z:
        manifest = json.loads(z.read('archive_hashes.json'))
        def checked(name):
            data = z.read(name)
            assert sha_bytes(data) == manifest[name], name
            return data
        inventory = {d['record_id']: d for d in json.loads(checked('resume_inventory.json'))['documents']}
        for rid in selected:
            d = inventory[rid]
            targets = {saved[(rid, e)]['attack_target'] for e in (2, 4, 8, 16) if (rid, e) in saved}
            assert len(targets) == 1 and d['clean_target'] == clean[rid]['clean_target']
            images = []
            for name in d['images']:
                put('pages/' + name, checked(name))
                images.append({'path': 'pages/' + name, 'sha256': hashes['pages/' + name]})
            assert len(images) == clean[rid]['page_count']
            documents.append({'record_id': rid, 'clean_target': d['clean_target'], 'attack_target': targets.pop(), 'images': images,
                'clean_recovery': status(clean[rid]['methods']['contextual_recovery']),
                'saved_standard': {str(e): {'undefended_targeted': bool(saved[(rid, e)]['methods']['none']['targeted']),
                                            'undefended': status(saved[(rid, e)]['methods']['none']),
                                            'recovery': status(saved[(rid, e)]['methods']['contextual_recovery'])} for e in (8, 16)}})
    local_zip.unlink()
    (temp / 'attack_plan.json').write_text(json.dumps({'selection': 'First ten reserved IDs with saved attacks at epsilon 8 and 16, in the frozen config order; not selected on outcomes.', 'documents': documents}, indent=2))
    (temp / 'hashes.json').write_text(json.dumps(hashes, indent=2))
    (temp / 'READY.json').write_text(json.dumps({'documents': len(documents)}))
    temp.rename(ATTACK_ROOT)
attack_config = json.loads((ATTACK_ROOT / 'config.json').read_text())
attack_plan = json.loads((ATTACK_ROOT / 'attack_plan.json').read_text())
print('Prepared resumes:', len(attack_plan['documents']))
for d in attack_plan['documents']:
    print(' ', d['record_id'].split('/')[-1], d['clean_target'], '->', d['attack_target'], '| pages', len(d['images']), '| clean recovery', d['clean_recovery'])


## 2. Install the recorded environment

In [ ]:
def attack_install(args):
    print(' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)
attack_install([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
ATTACK_PYTHON = ATTACK_ROOT / 'venv/bin/python'
if not ATTACK_PYTHON.exists():
    attack_install(['uv', 'venv', '--python', attack_config['python'], '--seed', ATTACK_ROOT / 'venv'])
v = attack_config['versions']
assert v['torch'].endswith('+cu124') and v['torchvision'].endswith('+cu124'), v
attack_install(['uv', 'pip', 'install', '--python', ATTACK_PYTHON, '--index-url', 'https://download.pytorch.org/whl/cu124',
                'torch==' + v['torch'], 'torchvision==' + v['torchvision']])
attack_install(['uv', 'pip', 'install', '--python', ATTACK_PYTHON] + [k + '==' + value for k, value in v.items() if k not in ('torch', 'torchvision')])
attack_install([ATTACK_PYTHON, '-m', 'pip', 'check'])


## 3. Write the attack runner

In [ ]:
RUNNER_SOURCE = '"""Bounded direct attack on the frozen v4 resume contextual-recovery rule. New study; the defense is unchanged."""\nimport argparse,hashlib,json,math,sys,time,os,platform,importlib.metadata,csv\nfrom pathlib import Path\nLOC=\'Find the education and qualifications sections on this resume page. Locate whole sections, including their headings, degree or diploma names, institutions, dates and any continuation lines. Include stated qualifications even if the page does not explicitly say completed or awarded. Include education entries outside a heading too. Do not infer qualifications from job titles, work experience or skills. Group nearby lines in the same section into one box rather than returning a box per word or line. Return only JSON in this exact format: {"boxes": [[left, top, right, bottom]]}. Use pixel coordinates on the displayed image of width {w} and height {h}. Return {"boxes": []} only if no education or qualification entries are visible. Return at most four section boxes. Do not include explanations or transcribed text.\'\nPOLICY={\'version\':\'resume_development_retry_v4\',\'localization_attempts\':2,\'minimum_height\':.25,\'margin\':.05,\'expansion\':.05,\'max_height\':.70,\'expanded_max_height\':.85,\'max_regions_per_page\':4,\'max_raw_boxes\':16,\'max_crops_per_document\':8,\'localization_tokens\':768,\'fixed_band\':[.2,.8]}\ndef read(p):return json.loads(Path(p).read_text())\ndef sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef write(p,x):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(\'.tmp\');t.write_text(json.dumps(x,indent=2,allow_nan=False));t.replace(p)\ndef parse(raw,processed,original):\n import re\n s=raw.strip()\n # Accept one fenced JSON response, even when preceded by explanation.\n fences=re.findall(r"```(?:json)?\\s*(.*?)```",s,flags=re.S|re.I)\n if fences:\n  if len(fences)!=1:raise ValueError(\'Ambiguous multiple JSON blocks\')\n  s=fences[0].strip()\n empty_phrases={\n  \'there are none.\',\n  \'there are no regions indicating completed education qualifications or awarded degrees on this resume page.\',\n  \'there are no regions stating completed education qualifications or awarded degrees on this resume page.\'}\n if s.lower() in empty_phrases:return []\n # Only repair trailing commas outside strings; do not extract arbitrary numbers.\n cleaned=[];quoted=False;escaped=False\n for i,ch in enumerate(s):\n  if quoted:\n   cleaned.append(ch)\n   if escaped:escaped=False\n   elif ch==\'\\\\\':escaped=True\n   elif ch==\'"\':quoted=False\n  else:\n   if ch==\'"\':quoted=True\n   if ch==\',\' and s[i+1:].lstrip().startswith((\']\', \'}\')):continue\n   cleaned.append(ch)\n obj=json.loads(\'\'.join(cleaned))\n def unpack(x):\n  if isinstance(x,dict):\n   schemas=[k for k in (\'boxes\',\'bbox_2d\') if k in x]\n   named=all(k in x for k in (\'left\',\'top\',\'right\',\'bottom\'))\n   if len(schemas)+int(named)!=1:raise ValueError(\'Unknown or ambiguous box schema\')\n   return unpack(x[schemas[0]]) if schemas else [[x[k] for k in (\'left\',\'top\',\'right\',\'bottom\')]]\n  if isinstance(x,list):\n   if len(x)==4 and all(isinstance(v,(int,float)) and not isinstance(v,bool) for v in x):return [x]\n   out=[]\n   for item in x:out.extend(unpack(item))\n   return out\n  raise ValueError(\'Invalid box representation\')\n vals=unpack(obj)\n # Bound raw output, but merge line boxes before limiting final regions.\n if len(vals)>POLICY[\'max_raw_boxes\']:raise ValueError(\'Too many raw boxes\')\n pw,ph=processed;w,h=original;out=[]\n for v in vals:\n  if len(v)!=4 or any(isinstance(x,bool) or not isinstance(x,(int,float)) or not math.isfinite(x) for x in v):raise ValueError(\'Invalid coordinates\')\n  l,t,r,b=v\n  if l>=r or t>=b:raise ValueError(\'Box endpoints inverted or zero area\')\n  if not (0<=l and r<=pw and 0<=t and b<=ph):raise ValueError(\'Box outside processed image\')\n  top,bottom=round(t*h/ph),round(b*h/ph)\n  if top>=bottom:raise ValueError(\'Box collapses when mapped to original image\')\n  out.append((0,top,w,bottom))\n return out\n\ndef views(boxes,size):\n w,h=size;intervals=[]\n for _,t,_,b in boxes:\n  height=max(round(.25*h),b-t+2*round(.05*h))\n  if height>.70*h:return None\n  top=max(0,min(h-height,round((t+b-height)/2)));intervals.append((top,top+height))\n merged=[]\n for t,b in sorted(intervals):\n  if merged and t<=merged[-1][1]:merged[-1]=(merged[-1][0],max(b,merged[-1][1]))\n  else:merged.append((t,b))\n if len(merged)>POLICY[\'max_regions_per_page\']:return None\n pairs=[]\n for t,b in merged:\n  t2=max(0,t-round(.05*h));b2=min(h,b+round(.05*h))\n  if b-t>.70*h or b2-t2>.85*h:return None\n  pairs.append(((0,t,w,b),(0,t2,w,b2)))\n return pairs\n\ndef localize(model,im,rs,grid):\n prompt=LOC.replace(\'{w}\',str(rs[0])).replace(\'{h}\',str(rs[1]))\n attempts=[];previous_tokens=model.max_new_tokens\n try:\n  model.max_new_tokens=POLICY[\'localization_tokens\']\n  for attempt in range(POLICY[\'localization_attempts\']):\n   query=prompt\n   if attempt:\n    query+=\' Your previous localization output was invalid. Inspect the page again and return fresh section boxes. Every box must satisfy 0 <= left < right <= \'+str(rs[0])+\' and 0 <= top < bottom <= \'+str(rs[1])+\'. Use positive-area boxes, group nearby education lines, and output only the requested JSON. Do not guess any qualification.\'\n   raw=model.predict([im],query)\n   assert model.last_input_info[\'image_grid_thw\']==grid\n   try:\n    boxes=parse(raw,rs,im.size);pairs=views(boxes,im.size)\n    if pairs is None:raise ValueError(\'Merged regions exceed size or count limits\')\n    attempts.append({\'raw\':raw,\'status\':\'valid\'})\n    return {\'raw\':raw,\'processed_size\':rs,\'boxes\':boxes,\'pairs\':pairs,\'status\':\'ok\' if pairs else \'no_education\',\'attempts\':attempts}\n   except (ValueError,KeyError,TypeError) as exc:\n    attempts.append({\'raw\':raw,\'status\':\'invalid\',\'error\':str(exc)})\n  return {\'raw\':attempts[-1][\'raw\'],\'processed_size\':rs,\'status\':\'invalid\',\'error\':attempts[-1][\'error\'],\'attempts\':attempts}\n finally:model.max_new_tokens=previous_tokens\n\n\n# ---------------------------------------------------------------------------------------------\n# Everything above this line is copied verbatim from the frozen v4 runner (see FROZEN_SOURCE_SHA256).\n# Everything below is new: a bounded direct attack on the frozen contextual-recovery rule.\n# ---------------------------------------------------------------------------------------------\nFROZEN_SOURCE_SHA256=\'b09b577474d73420f4f6891bd70ba4a486bbed1c40005d95439866be1e50b844\'\nATTACK={\'version\':\'resume_direct_recovery_v1\',\'documents\':10,\'epsilons\':[8,16],\'restarts\':[0,1],\n        \'steps\':100,\'refresh_interval\':10,\'queries\':40,\'step_fraction\':0.25,\'seed\':566,\n        \'wall_seconds\':14400,\'run_margin_seconds\':1500,\'early_stop_on_accepted_target\':True,\n        \'processor_tolerance\':{\'max\':1e-6,\'mean\':1e-8,\'surrogate_mean\':0.05},\n        \'objective\':\'cross-entropy of the target label tokens on the localized views and on the expanded views; boxes held fixed between verifier queries\',\n        \'order\':\'all restart-0 runs first, then restart-1; inside a restart by frozen document order, then epsilon\'}\n\ndef digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True).encode()).hexdigest()\n\ndef project(array,clean,eps):\n    import numpy as np\n    return np.clip(np.rint(array),np.maximum(0,clean.astype(float)-eps),np.minimum(255,clean.astype(float)+eps)).astype(\'uint8\')\n\ndef rank_of(obs,target):\n    return (int(obs[\'prediction\']==target),int(obs[\'crop\']==target)+int(obs[\'wide\']==target))\n\ndef run_attack(clean_pages,target,eps,restart,seed_key,observe_fn,step_fn,settings):\n    """clean_pages: uint8 HxWx3 arrays. observe_fn(arrays)->observation. step_fn(arrays,observation)->(arrays,loss)."""\n    import numpy as np\n    rng=np.random.default_rng(int(seed_key[:8],16)+restart)\n    refreshes=(settings[\'steps\']+settings[\'refresh_interval\']-1)//settings[\'refresh_interval\']\n    assert settings[\'queries\']>=refreshes+2,\'query budget cannot cover refreshes and final verification\'\n    current=[project(c.astype(float)+rng.integers(-eps,eps+1,c.shape),c,eps) for c in clean_pages]\n    state={\'queries\':0,\'step\':0,\'trace\':[],\'best\':None,\'best_rank\':None}\n    def query(arrays,phase,loss=None,strict=False):\n        obs=observe_fn(arrays);r=rank_of(obs,target);state[\'queries\']+=1\n        state[\'trace\'].append({\'phase\':phase,\'step\':state[\'step\'],\'query\':state[\'queries\'],\'prediction\':obs[\'prediction\'],\n            \'crop\':obs[\'crop\'],\'wide\':obs[\'wide\'],\'localization_valid\':obs[\'valid\'],\'crops\':len(obs[\'boxes\']),\'surrogate_loss\':loss})\n        better=state[\'best_rank\'] is None or (r>state[\'best_rank\'] if strict else r>=state[\'best_rank\'])\n        if better:state[\'best\']=[a.copy() for a in arrays];state[\'best_rank\']=r\n        return obs\n    usable=query(current,\'initial\')\n    stop=lambda:settings[\'early_stop_on_accepted_target\'] and state[\'best_rank\'][0]==1\n    while state[\'step\']<settings[\'steps\'] and not stop():\n        current,loss=step_fn(current,usable)\n        current=[project(a,c,eps) for a,c in zip(current,clean_pages)]\n        state[\'step\']+=1\n        if state[\'step\']%settings[\'refresh_interval\']==0 or state[\'step\']==settings[\'steps\']:\n            obs=query(current,\'gradient\',loss)\n            if obs[\'boxes\']:usable=obs  # keep the last usable regions when localization fails\n    # Reserve the last verifier query for the saved, reloaded pages.\n    while state[\'queries\']<settings[\'queries\']-1 and not stop():\n        cand=[a.copy() for a in state[\'best\']];j=int(rng.integers(0,len(cand)));h,w=cand[j].shape[:2]\n        size=max(1,round(min(h,w)*(0.15 if state[\'queries\']<settings[\'queries\']//2 else 0.05)))\n        y=int(rng.integers(0,h-size+1));x=int(rng.integers(0,w-size+1));signs=rng.choice([-1,1],size=(1,1,3))\n        patch=clean_pages[j][y:y+size,x:x+size]\n        cand[j][y:y+size,x:x+size]=project(patch.astype(float)+eps*signs,patch,eps)\n        query(cand,\'query_search\',strict=True)\n    return state\n\ndef observe(model,pages,question,norm):\n    """Frozen v4 recovery rule on the current pages (same order of calls as the frozen runner)."""\n    regions=[];boxes=[];valid=True\n    for j,im in enumerate(pages):\n        proc=model.processor.image_processor(images=[im],return_tensors=\'pt\');grid=proc[\'image_grid_thw\'].tolist();assert len(grid)==1\n        _,gh,gw=grid[0];patch=model.processor.image_processor.patch_size;rs=(gw*patch,gh*patch);del proc\n        region=localize(model,im,rs,grid);region[\'page\']=j;regions.append(region)\n        if region[\'status\']==\'invalid\':\n            valid=False;continue\n        for b,e in region[\'pairs\']:boxes.append((j,tuple(b),tuple(e)))\n    raw1=raw2=None\n    if valid and 0<len(boxes)<=POLICY[\'max_crops_per_document\']:\n        raw1=model.predict([pages[j].crop(b) for j,b,_ in boxes],question)\n        raw2=model.predict([pages[j].crop(e) for j,_,e in boxes],question)\n    crop,wide=norm(raw1),norm(raw2)\n    return {\'regions\':regions,\'boxes\':boxes if valid else [],\'valid\':valid,\'raw_localized\':raw1,\'raw_expanded\':raw2,\n            \'crop\':crop,\'wide\':wide,\'prediction\':crop if crop is not None and crop==wide else None}\n\ndef summarize(out,plan):\n    results=[read(p) for p in sorted((out/\'runs\').glob(\'*/result.json\'))]\n    docs=plan[\'documents\'];rows=[]\n    for eps in ATTACK[\'epsilons\']:\n        done={};hit={}\n        for r in results:\n            if r[\'epsilon_255\']!=eps:continue\n            done.setdefault(r[\'record_id\'],set()).add(r[\'restart\'])\n            m=r[\'methods\'][\'contextual_recovery\'];hit[r[\'record_id\']]=hit.get(r[\'record_id\'],False) or bool(m[\'accepted\'] and m[\'targeted\'])\n        attempted=[d for d in docs if done.get(d[\'record_id\'])]\n        both=[d for d in docs if done.get(d[\'record_id\'])==set(ATTACK[\'restarts\'])]\n        eligible=[d for d in attempted if d[\'clean_recovery\']==\'correct\']\n        runs=[r for r in results if r[\'epsilon_255\']==eps]\n        rows.append({\'epsilon_255\':eps,\'documents_planned\':len(docs),\'documents_with_any_restart\':len(attempted),\'documents_with_all_restarts\':len(both),\n            \'success_documents_all\':sum(hit[d[\'record_id\']] for d in attempted),\'eligible_documents\':len(eligible),\n            \'success_documents_eligible\':sum(hit[d[\'record_id\']] for d in eligible),\'runs_completed\':len(runs),\n            \'runs_succeeded\':sum(bool(r[\'methods\'][\'contextual_recovery\'][\'accepted\'] and r[\'methods\'][\'contextual_recovery\'][\'targeted\']) for r in runs),\n            \'runs_rejected\':sum(not r[\'methods\'][\'contextual_recovery\'][\'accepted\'] for r in runs),\n            \'runs_correct\':sum(bool(r[\'methods\'][\'contextual_recovery\'][\'correct\']) for r in runs),\n            \'saved_standard_attack_undefended_targeted\':sum(bool(d[\'saved_standard\'][str(eps)][\'undefended_targeted\']) for d in docs),\n            \'success_ids\':[d[\'record_id\'] for d in attempted if hit[d[\'record_id\']]]})\n    planned=len(docs)*len(ATTACK[\'epsilons\'])*len(ATTACK[\'restarts\'])\n    summary={\'complete\':len(results)==planned,\'runs_completed\':len(results),\'runs_planned\':planned,\'groups\':rows,\n        \'note\':\'Bounded direct attack on frozen v4 resume contextual recovery. First ten reserved resumes with saved attacks in the frozen order, not selected on outcomes. Missing runs are not attack failures. A document counts as a success if any completed restart yields an accepted target on the saved, reloaded pages. Not a robustness guarantee.\'}\n    write(out/\'summary.json\',summary)\n    with (out/\'summary.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=[k for k in rows[0] if k!=\'success_ids\']);w.writeheader()\n        for row in rows:w.writerow({k:v for k,v in row.items() if k!=\'success_ids\'})\n    print(json.dumps(summary,indent=2),flush=True)\n    return summary\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--out\',required=True);a=p.parse_args();root=Path(a.root);out=Path(a.out)\n    hashes=read(root/\'hashes.json\')\n    for name,h in hashes.items():assert sha(root/name)==h,name\n    c=read(root/\'config.json\');plan=read(root/\'attack_plan.json\');docs=plan[\'documents\']\n    assert len(docs)==ATTACK[\'documents\'] and len({d[\'record_id\'] for d in docs})==len(docs)\n    assert {d[\'record_id\'] for d in docs}<=set(c[\'reserved_ids\']) and not {d[\'record_id\'] for d in docs}&set(c[\'development_ids\'])\n    for d in docs:assert d[\'attack_target\'] in c[\'labels\'] and d[\'attack_target\']!=d[\'clean_target\']\n    for k,v in c[\'versions\'].items():assert importlib.metadata.version(k)==v,(k,v)\n    assert platform.python_version()==c[\'python\']\n    os.environ[\'HF_HOME\']=str(root/\'hf_cache\')\n    import numpy as np,torch\n    from PIL import Image\n    sys.path.insert(0,str(root/\'src\'))\n    from attacks.qwen import Objective,clear_rope\n    from models.donut import rgb_tensor_to_pil\n    from models.qwen import QwenVL\n    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n    contract={\'inputs\':sha(root/\'hashes.json\'),\'plan\':sha(root/\'attack_plan.json\'),\'frozen_rule_source_sha256\':FROZEN_SOURCE_SHA256,\'script\':sha(__file__),\n              \'policy\':POLICY,\'prompt\':LOC,\'attack\':ATTACK,\'versions\':c[\'versions\'],\'model\':c[\'model\'],\'question\':c[\'question\']}\n    out.mkdir(parents=True,exist_ok=True)\n    if (out/\'contract.json\').exists():assert read(out/\'contract.json\')==contract,\'Changed inputs/environment/code: use a new output directory.\'\n    else:write(out/\'contract.json\',contract)\n    sig=digest(contract)\n    sessions=read(out/\'sessions.json\') if (out/\'sessions.json\').exists() else []\n    sessions.append({\'started\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\'gpu\':torch.cuda.get_device_name(),\'cuda\':torch.version.cuda});write(out/\'sessions.json\',sessions)\n    torch.manual_seed(ATTACK[\'seed\'])\n    model=QwenVL(device=torch.device(\'cuda\'),**c[\'model\'])\n    raw_predict=model.predict\n    def predict(images,prompt):\n        clear_rope(model.model)  # gradient forwards leave cached rotary offsets\n        return raw_predict(images,prompt)\n    model.predict=predict\n    norm=lambda x:x.strip().lower() if isinstance(x,str) and x.strip().lower() in c[\'labels\'] else None\n    question=c[\'question\'];tokenizer=model.processor.tokenizer\n    observe_fn=lambda arrays:observe(model,[Image.fromarray(x) for x in arrays],question,norm)\n    def make_step(target,eps,diagnostics):\n        cache={}\n        def objective(views):\n            obj=Objective(model,[rgb_tensor_to_pil(v) for v in views],question,target,ATTACK[\'processor_tolerance\'])\n            ids=tokenizer.encode(target,add_special_tokens=False);assert ids\n            tail=torch.tensor([ids],device=model.device,dtype=torch.long);prefix=obj.prefix[\'input_ids\']\n            obj.ids=torch.cat([prefix,tail],dim=1)\n            obj.positions=torch.arange(prefix.shape[1]-1,obj.ids.shape[1]-1,device=model.device);obj.target_tensor=tail.flatten()\n            with torch.no_grad():stats=obj.parity(views)\n            diagnostics[\'surrogate_mean_max\']=max(diagnostics[\'surrogate_mean_max\'],stats[\'processor_surrogate_mean_abs_difference\'])\n            return obj\n        def step(arrays,obs):\n            xs=[torch.from_numpy(x.astype(\'float32\')/255).permute(2,0,1).contiguous().to(model.device).requires_grad_(True) for x in arrays]\n            boxes=obs[\'boxes\']\n            groups=[[(j,b) for j,b,_ in boxes],[(j,e) for j,_,e in boxes]] if boxes else [[(j,None) for j in range(len(arrays))]]\n            diagnostics[\'steps_on_regions\' if boxes else \'steps_on_full_pages\']+=1\n            grads=[torch.zeros_like(x) for x in xs];total=0.\n            for group in groups:\n                views=[xs[j] if box is None else xs[j][:,box[1]:box[3],box[0]:box[2]] for j,box in group]\n                key=tuple((j,box,tuple(v.shape)) for (j,box),v in zip(group,views))\n                if key not in cache:\n                    if len(cache)>=4:cache.clear()\n                    cache[key]=objective(views)\n                loss,_=cache[key].loss(views)\n                used=sorted({j for j,_ in group})\n                gs=torch.autograd.grad(loss,[xs[j] for j in used])\n                assert torch.isfinite(loss)\n                for j,g in zip(used,gs):\n                    assert torch.isfinite(g).all();grads[j]+=g.detach()\n                total+=float(loss.detach());del loss,gs,views\n            assert any(bool(g.abs().max()>0) for g in grads),\'Zero gradient\'\n            with torch.no_grad():\n                updated=[np.asarray(rgb_tensor_to_pil(x-(eps/255)*ATTACK[\'step_fraction\']*g.sign())) for x,g in zip(xs,grads)]\n            del xs,grads\n            return updated,total\n        return step\n    schedule=[(restart,d,eps) for restart in ATTACK[\'restarts\'] for d in docs for eps in ATTACK[\'epsilons\']]\n    started=time.monotonic()\n    try:\n        for index,(restart,d,eps) in enumerate(schedule):\n            key=digest([d[\'record_id\'],eps,restart]);directory=out/\'runs\'/key;dest=directory/\'result.json\'\n            if dest.exists():\n                print(f\'DONE {index+1}/{len(schedule)} {d["record_id"]} eps={eps} restart={restart}\',flush=True);continue\n            if time.monotonic()-started>ATTACK[\'wall_seconds\']-ATTACK[\'run_margin_seconds\']:\n                print(\'Session budget reached before starting the next run. Rerun the notebook to continue.\',flush=True);break\n            for im in d[\'images\']:assert sha(root/im[\'path\'])==im[\'sha256\'],im[\'path\']\n            clean=[np.asarray(Image.open(root/x[\'path\']).convert(\'RGB\')) for x in d[\'images\']]\n            print(f\'ATTACK {index+1}/{len(schedule)} {d["record_id"]} eps={eps} restart={restart} target={d["attack_target"]}\',flush=True)\n            t0=time.monotonic();diagnostics={\'surrogate_mean_max\':0.,\'steps_on_regions\':0,\'steps_on_full_pages\':0}\n            state=run_attack(clean,d[\'attack_target\'],eps,restart,digest([sig,d[\'record_id\'],eps,restart]),observe_fn,make_step(d[\'attack_target\'],eps,diagnostics),ATTACK)\n            directory.mkdir(parents=True,exist_ok=True);images=[];reloaded=[];linf=0\n            for j,(best,original) in enumerate(zip(state[\'best\'],clean)):\n                path=directory/f\'page_{j:02d}.png\';Image.fromarray(best).save(path)\n                with Image.open(path) as im:arr=np.asarray(im.convert(\'RGB\')).copy()\n                assert arr.shape==original.shape\n                linf=max(linf,int(np.abs(arr.astype(np.int16)-original.astype(np.int16)).max()))\n                reloaded.append(arr);images.append({\'path\':str(path.relative_to(out)),\'sha256\':sha(path)})\n            assert linf<=eps,\'pixel budget exceeded\'\n            final=observe_fn(reloaded);queries=state[\'queries\']+1\n            fullraw=model.predict([Image.fromarray(x) for x in reloaded],question);full=norm(fullraw)\n            preds={\'none\':full,\'localized_only\':final[\'crop\'],\'full_region_agreement\':full if full is not None and full==final[\'crop\'] else None,\'contextual_recovery\':final[\'prediction\']}\n            result={\'contract\':sig,\'record_id\':d[\'record_id\'],\'clean_target\':d[\'clean_target\'],\'attack_target\':d[\'attack_target\'],\'method_attacked\':\'contextual_recovery\',\n                \'epsilon_255\':eps,\'restart\':restart,\'steps\':state[\'step\'],\'queries\':queries,\'runtime_seconds\':time.monotonic()-t0,\'linf_255\':linf,\'images\':images,\n                \'candidate_rank\':list(state[\'best_rank\']),\'saved_png_matches_candidate\':int(final[\'prediction\']==d[\'attack_target\'])==state[\'best_rank\'][0],\n                \'methods\':{k:{\'prediction\':v,\'correct\':v is not None and v==d[\'clean_target\'],\'accepted\':v is not None,\'targeted\':v is not None and v==d[\'attack_target\']} for k,v in preds.items()},\n                \'raw_answers\':{\'full\':fullraw,\'localized\':final[\'raw_localized\'],\'expanded\':final[\'raw_expanded\']},\'regions\':final[\'regions\'],\'crop_count\':len(final[\'boxes\']),\n                \'localization_valid\':final[\'valid\'],\'diagnostics\':diagnostics,\'trace\':state[\'trace\'],\'clean_recovery\':d[\'clean_recovery\'],\n                \'attack_note\':\'Bounded surrogate PGD on localized and expanded views plus verifier query search. Boxes fixed between verifier queries. Not an exhaustive adaptive attack or a robustness certificate.\'}\n            write(dest,result)\n            print(f\'SAVED {d["record_id"]} eps={eps} restart={restart} recovery={preds["contextual_recovery"]} full={full} steps={state["step"]} queries={queries} seconds={result["runtime_seconds"]:.0f}\',flush=True)\n            torch.cuda.empty_cache()\n        summarize(out,plan)\n    finally:model.unload()\nif __name__==\'__main__\':main()\n'
ATTACK_RUNNER = ATTACK_ROOT / 'resume_direct_attack.py'
ATTACK_RUNNER.write_text(RUNNER_SOURCE)
assert hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest() == 'be215efff96467ea4fb9b552bb721698f71a0bbc86b4677eda26f24a39a4b1b4'
print('Runner written:', ATTACK_RUNNER)


## 4. Run the attack (resumable)

In [ ]:
if 'process' in globals():
    assert process.poll() is not None, 'A previous run is still active.'
ATTACK_OUT.mkdir(parents=True, exist_ok=True)
attack_log = ATTACK_OUT / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.log')
attack_env = os.environ.copy()
attack_env['MPLBACKEND'] = 'Agg'
attack_env['HF_HOME'] = str(ATTACK_ROOT / 'hf_cache')
with attack_log.open('w') as log:
    process = subprocess.Popen([str(ATTACK_PYTHON), '-u', str(ATTACK_RUNNER), '--root', str(ATTACK_ROOT), '--out', str(ATTACK_OUT)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=attack_env)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        status_code = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
assert status_code == 0, f'Run failed; share this log: {attack_log}'


## 5. Summary and export

In [ ]:
summary = json.loads((ATTACK_OUT / 'summary.json').read_text())
print('Runs completed:', summary['runs_completed'], 'of', summary['runs_planned'], '| complete:', summary['complete'])
for g in summary['groups']:
    print(f"eps {g['epsilon_255']}: resumes attacked {g['documents_with_any_restart']}/10 (both restarts {g['documents_with_all_restarts']}) | "
          f"success all {g['success_documents_all']}/{g['documents_with_any_restart']} | eligible {g['success_documents_eligible']}/{g['eligible_documents']} | "
          f"runs succeeded {g['runs_succeeded']}/{g['runs_completed']} | undefended saved standard attack {g['saved_standard_attack_undefended_targeted']}/10")
archive = RESUME_DRIVE / 'resume_direct_attack_v1.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(ATTACK_OUT.rglob('*')):
        if f.is_file() and f.suffix in ('.json', '.csv', '.png', '.log'):
            z.write(f, f.relative_to(ATTACK_OUT))
print('Share this file:', archive)
if not summary['complete']:
    print('Not all 40 runs are done. Use Runtime > Run all again to continue; finished runs are kept.')
from google.colab import files
files.download(str(archive))
